In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from IPython.display import display, Markdown

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

In [2]:
importr("CASdatasets")

rpy2.robjects.packages.Package as a <module 'CASdatasets'>

In [28]:
name='fretelematic'
ro.r(f"data({name})")
with localconverter(ro.default_converter + pandas2ri.converter):
    df = ro.conversion.rpy2py(ro.r[name])
# R factors arrive as pandas Categorical -- convert to plain strings
for col in df.select_dtypes("category").columns:
    df[col] = df[col].astype(str)

In [30]:
df['Claim']=df['Claim'].map({'yes': 1, 'no': 0})

In [31]:
df.reset_index(drop=True, inplace=True)

In [32]:
df.drop(columns=['Policy_ID'], inplace=True)

In [33]:
df

,Total_Distance,Drive_Score,Time_Day,Style_Score,Corner_Score,Acceleration_Score,Braking_Score,Total_Night_Time,Total_Time,Acceleration,Brake,Corner,Insured_Gender,Insured_Age,Claim
0,3649.0,67.0,109.0,85.0,86.0,88.0,84.0,34.0,79.0,Low,Low,High,M,21.0,0
1,9066.0,63.0,116.0,92.0,93.0,80.0,93.0,28.0,339.0,High,High,High,F,20.0,0
2,27156.0,74.0,73.0,55.0,61.0,49.0,64.0,24.0,464.0,Low,High,High,F,30.0,0
3,8558.0,77.0,108.0,74.0,67.0,91.0,82.0,25.0,437.0,High,High,High,M,52.0,0
4,17853.0,68.0,106.0,65.0,61.0,89.0,72.0,38.0,649.0,High,High,High,M,21.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1172,25048.0,78.0,120.0,26.0,25.0,16.0,38.0,34.0,559.0,Low,High,Low,M,55.0,0
1173,7330.0,91.0,101.0,36.0,26.0,54.0,54.0,22.0,199.0,Low,High,Low,M,47.0,0
1174,7378.0,84.0,44.0,62.0,61.0,45.0,78.0,22.0,255.0,Low,High,High,M,61.0,1
1175,7176.0,68.0,116.0,95.0,91.0,99.0,97.0,27.0,261.0,High,High,High,F,48.0,0


In [34]:
X = df.drop(columns=['Claim'])
y = df['Claim']

In [35]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [45]:
from tabpfn import TabPFNClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
n_estimators_list = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
for n_estimators in n_estimators_list:
    mod = TabPFNClassifier(n_estimators=n_estimators)
    mod.fit(X_train.values, y_train.values)
    y_pred = mod.predict_proba(X_test.values)[:,1]
    print("ROC AUC Score:", roc_auc_score(y_test, y_pred))

ROC AUC Score: 0.4875919117647059
ROC AUC Score: 0.5153186274509804
ROC AUC Score: 0.498468137254902
ROC AUC Score: 0.5237438725490197
ROC AUC Score: 0.5471813725490196
ROC AUC Score: 0.5649509803921569
ROC AUC Score: 0.5381433823529412
